<a href="https://colab.research.google.com/github/rgsrinidhi/MLprojects/blob/main/LoRA_toy_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture

!pip install unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B",   # or "unsloth/Llama-3.2-1B" if you prefer
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [15]:
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

In [4]:
FastLanguageModel.for_inference(model)

instruction = "Give 3 tips for better sleep"

prompt = alpaca_prompt.format(instruction,"")

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, use_cache = True)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give 3 tips for better sleep

### Response:
1. **Establish a regular sleep schedule**: Aim for a consistent sleep schedule, ideally around the same time every day. This helps regulate your body's internal clock and ensures you get enough rest.
2. **Create a relaxing bedtime routine**: Go to bed at a comfortable temperature, turn off electronic devices, and engage in activities that promote relaxation, such as reading a book or listening to soft music.
3. **Avoid screens before bed**: Exposure to blue light from electronic devices can disrupt your sleep cycle. Try to avoid screens at least 3 hours before bedtime, and consider using apps like the Light Therapy app to help your body produce more melatonin naturally.


Find math related datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(len(dataset))
print(dataset[0])

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

52002
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [6]:
math_keywords = ["math", "calculate", "solve", "equation", "sum", "multiply", "divide", "subtract", "add", "percentage"]

def is_math_like(example):
    text = example["instruction"].lower()
    return any(kw in text for kw in math_keywords)

math_subset = dataset.filter(is_math_like)
print(math_subset[0])
print(len(math_subset))

Filter:   0%|          | 0/52002 [00:00<?, ? examples/s]

{'instruction': 'Identify the lines of longitude that divides North and South America.', 'input': '', 'output': 'The lines of longitude that divides North and South America is the line of the Panama Canal. This line of longitude cuts through the Isthmus of Panama and separates South America from Central America. It is approximately 800 km (500 mi) in total length and was completed in 1914. The Panama Canal is an important shipping route and serves as an international waterway connecting the Caribbean Sea with the Pacific Ocean.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nIdentify the lines of longitude that divides North and South America.\n\n### Response:\nThe lines of longitude that divides North and South America is the line of the Panama Canal. This line of longitude cuts through the Isthmus of Panama and separates South America from Central America. It is approximately 800 km (500 mi) in 

Change dataset because of lack of relevance

GSM8K Dataset


In [4]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main", split="train")
print(len(gsm8k))
print(gsm8k[0])

7473
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [5]:
import os

OUTPUT_FILE = "math_reformatted.txt"
progress = {"next_idx": 0}  # tracks where you left off in gsm8k

# Create the file fresh if it doesn't exist yet
if not os.path.exists(OUTPUT_FILE):
    open(OUTPUT_FILE, "w").close()
    print("Created new file.")
else:
    print("File already exists - will append to it.")

Created new file.


In [2]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main", split="train")



README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [15]:
batch_size = 30
start_idx = progress["next_idx"]
batch = gsm8k[start_idx : start_idx + batch_size]

for i in range(batch_size):
    q = batch["question"][i]
    a = batch["answer"][i]
    print(f"--- Problem {start_idx + i + 1} ---")
    print(f"Question: {q}")
    print(f"Raw Answer: {a}")
    print()

print(f"\n(This batch covers GSM8K indices {start_idx} to {start_idx + batch_size - 1})")

--- Problem 91 ---
Question: Herman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?
Raw Answer: December has 31 days, January has 31 days and February has 28 days for a total of 31+31+28 = <<31+31+28=90>>90 days
He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon for a total of 1/2+1/2 = <<1/2+1/2=1>>1 cup per day
If he feeds them 1 cup per day for 90 days then he will need 1*90 = <<1*90=90>>90 cups of birdseed
#### 90

--- Problem 92 ---
Question: John works a job that offers performance bonuses.  He makes $80 a day and works for 8 hours.  He has the option of working hard to earn the performance bonus of an extra $20 a day, but the extra effort results in a 2-hour longer workday.  How much does John make per hour if he decides to earn the bonus?
Raw Answer: First, we need to determine the length of John's workday if he decides to earn 

In [16]:
# Paste the LLM's full response here as a triple-quoted string
llm_output = """
Given:
Feeding months = December, January, February
December days = 31
January days = 31
February days = 28
Morning food = 1/2 cup
Afternoon food = 1/2 cup

Find:
Total cups of food needed

Solution:

1. Total days = 31 + 31 + 28 = 90
2. Food per day = 1/2 + 1/2 = 1 cup
3. Total food needed = 1 x 90 = 90 cups

Answer: 90

===

Given:
Regular daily pay = $80
Regular workday = 8 hours
Bonus = $20 per day
Extra work time = 2 hours

Find:
John's hourly pay if he earns the bonus

Solution:

1. Workday with bonus = 8 + 2 = 10 hours
2. Total daily pay = 80 + 20 = $100
3. Hourly pay = 100 / 10 = $10

Answer: 10

===

Given:
Sally's daily earnings = $6
Bob's daily earnings = $4
Each saves half of their earnings
Days in a year = 365

Find:
Total money Sally and Bob save in a year

Solution:

1. Sally's daily savings = 6 / 2 = $3
2. Sally's yearly savings = 3 x 365 = $1095
3. Bob's daily savings = 4 / 2 = $2
4. Bob's yearly savings = 2 x 365 = $730
5. Total savings = 1095 + 730 = $1825

Answer: 1825

===

Given:
Beef = 1000 pounds at $8 per pound
Chicken = twice as much beef at $3 per pound

Find:
Total cost of beef and chicken

Solution:

1. Beef cost = 8 x 1000 = $8000
2. Chicken = 1000 x 2 = 2000 pounds
3. Chicken cost = 2000 x 3 = $6000
4. Total cost = 8000 + 6000 = $14000

Answer: 14000

===

Given:
Pages per day = 20
Books = 3
Pages per book = 400

Find:
Number of days needed to write the books

Solution:

1. Total pages = 3 x 400 = 1200
2. Days needed = 1200 / 20 = 60

Answer: 60

===

Given:
Quarters = 20
Nickels per quarter = 5
Iron nickels = 20% of all nickels
Value of each iron nickel = $3
Value of each regular nickel = $0.05

Find:
Total value of Alice's money

Solution:

1. Nickels received = 20 x 5 = 100
2. Iron nickels = 100 x 20 / 100 = 20
3. Regular nickels = 100 - 20 = 80
4. Value of iron nickels = 20 x 3 = $60
5. Value of regular nickels = 80 x 0.05 = $4
6. Total value = 60 + 4 = $64

Answer: 64

===

Given:
Rainfall Monday = 4 inches
Rainfall Tuesday = 3 inches
Water collected per inch = 15 gallons
Selling price = $1.20 per gallon

Find:
Money James made from selling the water

Solution:

1. Total rainfall = 4 + 3 = 7 inches
2. Water collected = 7 x 15 = 105 gallons
3. Money earned = 105 x 1.20 = $126

Answer: 126

===

Given:
History books = 12
Romance books = 8
Poetry books = 4
Western novels = 5
Biographies = 6
Half of bottom-section books are mystery books

Find:
Total number of books on the cart

Solution:

1. Mystery books = 5 + 6 = 11
2. Total books = 12 + 8 + 4 + 11 + 5 + 6 = 46

Answer: 46

===

Given:
Gumballs given to Todd = 4
Gumballs given to Alisha = twice Todd's amount
Gumballs given to Bobby = 5 less than four times Alisha's amount
Gumballs remaining = 6

Find:
Total gumballs Hector purchased

Solution:

1. Gumballs given to Alisha = 4 x 2 = 8
2. Gumballs given to Bobby = 8 x 4 - 5 = 27
3. Total gumballs purchased = 4 + 8 + 27 + 6 = 45

Answer: 45

===

Given:
First tank water = 48 gallons
First tank is twice the size of second tank
Second tank fish = 2 inches
First tank fish = 3 inches
One first-tank fish eats another

Find:
How many more fish are in the first tank

Solution:

1. Second tank capacity = 48 / 2 = 24 gallons
2. Fish in second tank = 24 / 2 = 12
3. Fish in first tank = 48 / 3 = 16
4. Fish remaining in first tank = 16 - 1 = 15
5. Difference = 15 - 12 = 3

Answer: 3

===

Given:
Total sales = 36
Fabric section = 1/3 of sales
Jewelry section = 1/4 of sales
Stationery section = remaining sales

Find:
Sales in the stationery section

Solution:

1. Fabric sales = 36 / 3 = 12
2. Jewelry sales = 36 / 4 = 9
3. Stationery sales = 36 - 12 - 9 = 15

Answer: 15

===

Given:
Lip gloss per tube = enough for 3 people
Tubs of lip gloss = 6
Tubes per tub = 2

Find:
Number of people Marcy is doing makeup for

Solution:

1. Tubes of lip gloss = 6 x 2 = 12
2. People served = 12 x 3 = 36

Answer: 36

===

Given:
Oil needed per wheel = 10 ml
Wheels = 2
Additional oil for rest of bike = 5 ml

Find:
Total oil needed

Solution:

1. Oil for wheels = 2 x 10 = 20 ml
2. Total oil needed = 20 + 5 = 25 ml

Answer: 25

===

Given:
Typing speed = 6 sentences per minute
Typing before break = 20 minutes
Typing after break = 15 minutes
Typing after meeting = 18 minutes
Sentences erased = 40
Final total sentences = 536

Find:
Sentences already typed before today

Solution:

1. Sentences typed before break = 6 x 20 = 120
2. Sentences typed after break = 6 x 15 = 90
3. Sentences typed after meeting = 6 x 18 = 108
4. Total sentences typed today = 120 + 90 + 108 = 318
5. Sentences remaining from today's typing = 318 - 40 = 278
6. Sentences typed before today = 536 - 278 = 258

Answer: 258

===

Given:
Boat trips per day = 4
People per trip = 12
Days = 2

Find:
Total people the boat can transport

Solution:

1. People per day = 4 x 12 = 48
2. People in 2 days = 48 x 2 = 96

Answer: 96

===

Given:
Washington's goats = 140
Paddington has 40 more goats than Washington

Find:
Total goats they have

Solution:

1. Paddington's goats = 140 + 40 = 180
2. Total goats = 140 + 180 = 320

Answer: 320

===

Given:
Magic card packs = 10
Cards per pack = 20
Uncommon cards = 1/4 of each pack

Find:
Total uncommon cards

Solution:

1. Uncommon cards per pack = 20 / 4 = 5
2. Total uncommon cards = 10 x 5 = 50

Answer: 50

===

Given:
Tuesday traffic = 25 cars
Monday traffic = 20% less than Tuesday
Wednesday traffic = 2 more than Monday
Thursday traffic = 10 cars
Friday traffic = 10 cars
Saturday traffic = 5 cars
Sunday traffic = 5 cars

Find:
Total cars from Monday through Sunday

Solution:

1. Monday's decrease = 25 x 20 / 100 = 5
2. Monday traffic = 25 - 5 = 20
3. Wednesday traffic = 20 + 2 = 22
4. Thursday and Friday traffic = 10 x 2 = 20
5. Weekend traffic = 5 x 2 = 10
6. Total traffic = 20 + 25 + 22 + 20 + 10 = 97

Answer: 97

===

Given:
Pills per day = 9
Days = 14
Pills costing $1.50 each = 4 per day
Other pills cost $5.50 more each

Find:
Total money spent on pills

Solution:

1. Other pills per day = 9 - 4 = 5
2. Cost of each other pill = 1.50 + 5.50 = $7
3. Cost of other pills per day = 7 x 5 = $35
4. Cost of first 4 pills per day = 1.50 x 4 = $6
5. Total daily cost = 35 + 6 = $41
6. Total cost for 14 days = 41 x 14 = $574

Answer: 574

===

Given:
Total chicken nuggets = 100
Keely eats twice as many as Alyssa
Kendall eats twice as many as Alyssa

Find:
Chicken nuggets Alyssa ate

Solution:

1. Let Alyssa's nuggets = A
2. Keely's nuggets = 2A
3. Kendall's nuggets = 2A
4. Total nuggets = A + 2A + 2A = 5A
5. Alyssa's nuggets = 100 / 5 = 20

Answer: 20

===

Given:
Candy bar price = $2
Marvin sold = 35 candy bars
Tina sold = 3 times Marvin's amount

Find:
How much more money Tina made than Marvin

Solution:

1. Marvin's sales = 35 x 2 = $70
2. Tina's sales = 70 x 3 = $210
3. Difference = 210 - 70 = $140

Answer: 140

===

Given:
Bread = $2
Butter = $3
Juice = twice the price of bread
Money available = $15

Find:
Money Paul has left

Solution:

1. Juice cost = 2 x 2 = $4
2. Total spent = 2 + 3 + 4 = $9
3. Money left = 15 - 9 = $6

Answer: 6

===

Given:
Photos in gallery = 400
First-day photos = half the gallery amount
Second-day photos = 120 more than first-day photos

Find:
Total photos after adding both days' photos

Solution:

1. First-day photos = 400 / 2 = 200
2. Gallery after first day = 400 + 200 = 600
3. Second-day photos = 200 + 120 = 320
4. Total photos = 600 + 320 = 920

Answer: 920

===

Given:
Initial marbles = 25
Marbles lost = 20%
Friend gives twice the remaining amount

Find:
Total marbles Baez ends up with

Solution:

1. Marbles lost = 25 x 20 / 100 = 5
2. Marbles remaining = 25 - 5 = 20
3. Marbles received from friend = 20 x 2 = 40
4. Total marbles = 20 + 40 = 60

Answer: 60

===

Given:
Dogs = 2
Cats = 3
Fish = twice the number of cats and dogs combined

Find:
Total pets Ed has

Solution:

1. Dogs and cats = 2 + 3 = 5
2. Fish = 5 x 2 = 10
3. Total pets = 5 + 10 = 15

Answer: 15

===

Given:
Coconut-flavored jelly beans = 750
Coconut-flavored beans = 1/4 of red jelly beans
Red jelly beans = 3/4 of all jelly beans

Find:
Total jelly beans in the jar

Solution:

1. Red jelly beans = 750 x 4 = 3000
2. Total jelly beans = 3000 / 3 x 4 = 4000

Answer: 4000

===

Given:
Thank you cards = 3
Bills = 2
Rebates = 3 more than bills
Job applications = twice the rebates
Electric bill needs = 2 stamps
All other mail needs = 1 stamp

Find:
Total stamps Valerie needs

Solution:

1. Thank you cards = 3 x 1 = 3 stamps
2. Rebates = 3 + 2 = 5
3. Job applications = 2 x 5 = 10
4. Total pieces of mail = 3 + 2 + 5 + 10 = 20
5. Extra stamp for electric bill = 20 + 1 = 21 stamps

Answer: 21

===

Given:
Total fish = 100
Clownfish = half of total fish
Blowfish = half of total fish
Blowfish remaining in own tank = 26
Clownfish join display tank in equal number to display-tank blowfish
One-third of display-tank clownfish swim back

Find:
Clownfish remaining in the display tank

Solution:

1. Blowfish = 100 / 2 = 50
2. Blowfish in display tank = 50 - 26 = 24
3. Clownfish swimming back = 24 / 3 = 8
4. Clownfish remaining in display tank = 24 - 8 = 16

Answer: 16

===

Given:
Starting objects juggled = 3
Additional objects per week = 2
Practice period = 5 weeks

Find:
Objects Valerie can juggle after 5 weeks

Solution:

1. Additional objects learned = 2 x 5 = 10
2. Total objects = 10 + 3 = 13

Answer: 13

===

Given:
Fourth-grade classes = 3
Students per fourth-grade class = 30
P.E. class = 50 students

Find:
Total cupcakes needed

Solution:

1. Cupcakes for fourth-grade classes = 3 x 30 = 90
2. Total cupcakes = 90 + 50 = 140

Answer: 140

"""

formatted_examples = [ex.strip() for ex in llm_output.split("===") if ex.strip()]

with open(OUTPUT_FILE, "a") as f:
    for ex in formatted_examples:
        f.write(ex + "\n===\n")  # keep the same separator in the saved file

progress["next_idx"] = start_idx + batch_size  # move the tracker forward

print(f"Appended {len(formatted_examples)} examples. File now tracks up to index {progress['next_idx']}.")

Appended 30 examples. File now tracks up to index 120.


In [5]:
content = open("math_reformatted.txt").read()  # or wherever you saved it
examples = [e.strip() for e in content.split("===") if e.strip()]

print("Reformatted examples:", len(examples))

# Sanity check a few indices - does reformatted[idx] plausibly match gsm8k[idx]?
for idx in [0, 1, 5, 50, 119]:
    print(f"--- idx {idx} ---")
    print("GSM8K Q:", gsm8k[idx]["question"][:100])
    print("Reformatted:", examples[idx][:100].replace("\n", " | "))
    print()

Reformatted examples: 120
--- idx 0 ---
GSM8K Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m
Reformatted: Given: | Clips sold in April = 48 | Clips sold in May = half of April's amount |  | Find: | Total clips sold i

--- idx 1 ---
GSM8K Q: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much 
Reformatted: Given: | Weng's hourly babysitting rate = $12 | Babysitting time = 50 minutes |  | Find: | Money Weng earned |  | 

--- idx 5 ---
GSM8K Q: Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are y
Reformatted: Given: | Yellow flowers = 10 | Purple flowers = 80% more than yellow | Green flowers = 25% of yellow and p

--- idx 50 ---
GSM8K Q: Gerald spends $100 a month on baseball supplies. His season is 4 months long. He wants to use the mo
Reformatted: Given: | Monthly baseball supplies = $100 | Baseball season = 4 months | Months not playi

In [7]:
# Build paired (instruction, response) dataset
import os
import json
OUTPUT_FILE_PAIRED = "math_reformatted_paired_data.json"

# Create the file fresh if it doesn't exist yet
if not os.path.exists(OUTPUT_FILE_PAIRED):
    open(OUTPUT_FILE_PAIRED, "w").close()
    print("Created new file.")
else:
    print("File already exists - will append to it.")


paired_data = []
for i in range(len(examples)):
    paired_data.append({
        "instruction": gsm8k[i]["question"],
        "response": examples[i]
    })

with open(OUTPUT_FILE_PAIRED, "a") as json_file:
    json.dump(paired_data, json_file, indent=4)

print(f"Total paired examples: {len(paired_data)}")
print()
print("--- Sample pair ---")
print("Instruction:", paired_data[0]["instruction"])
print()
print("Response:", paired_data[0]["response"])

Created new file.
Total paired examples: 120

--- Sample pair ---
Instruction: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Response: Given:
Clips sold in April = 48
Clips sold in May = half of April's amount

Find:
Total clips sold in April and May

Solution:

1. Clips sold in May = 48 / 2 = 24
2. Total = 48 + 24 = 72

Answer: 72


In [8]:
# Train/test split: 100 train, 20 held-out
train_data = paired_data[:100]
test_data = paired_data[100:120]

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Train: 100, Test: 20


In [9]:
import re

# Check structural consistency across all 120 examples
issues = []
for i, ex in enumerate(paired_data):
    resp = ex["response"]

    # Check all 4 sections are present, in order
    has_given = "Given:" in resp
    has_find = "Find:" in resp
    has_solution = "Solution:" in resp
    has_answer = "Answer:" in resp

    if not (has_given and has_find and has_solution and has_answer):
        issues.append((i, "missing a section"))
        continue

    # Check order is correct
    order = [resp.index("Given:"), resp.index("Find:"), resp.index("Solution:"), resp.index("Answer:")]
    if order != sorted(order):
        issues.append((i, "sections out of order"))

    # Check for stray calculator annotations that might have leaked through
    if "<<" in resp or ">>" in resp:
        issues.append((i, "leftover calculator annotation"))

print(f"Checked {len(paired_data)} examples, found {len(issues)} issues")
for idx, problem in issues[:20]:
    print(f"  idx {idx}: {problem}")

Checked 120 examples, found 0 issues


In [10]:
#EOS Toke
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
151643


In [11]:
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # "<|endoftext|>"

def format_training_example(instruction, response):
    return alpaca_prompt.format(instruction, response) + EOS_TOKEN

In [12]:
sample = format_training_example(train_data[0]["instruction"], train_data[0]["response"])
print(sample)
print("---")
print("Ends with EOS:", sample.endswith(EOS_TOKEN))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

### Response:
Given:
Clips sold in April = 48
Clips sold in May = half of April's amount

Find:
Total clips sold in April and May

Solution:

1. Clips sold in May = 48 / 2 = 24
2. Total = 48 + 24 = 72

Answer: 72<|endoftext|>
---
Ends with EOS: True


In [13]:
from datasets import Dataset

formatted_texts = [
    format_training_example(ex["instruction"], ex["response"])
    for ex in train_data
]

train_dataset = Dataset.from_dict({"text": formatted_texts})

print(train_dataset)
print()
print(train_dataset[0]["text"][:200])  # quick peek

Dataset({
    features: ['text'],
    num_rows: 100
})

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Natalia sold clips to 48 of her friends in April, and then she sold half as 


In [ ]:
from huggingface_hub import login
login()

In [16]:
from google.colab import drive
drive.mount('/content/drive')

train_dataset.save_to_disk('/content/drive/MyDrive/solstice_project/train_dataset')

Mounted at /content/drive


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [16]:
train_dataset.push_to_hub("cheenshf/math_solution_FT_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 34.3kB / 34.3kB            

CommitInfo(commit_url='https://huggingface.co/datasets/cheenshf/math_solution_FT_dataset/commit/8e9b9d74716c7135b62d2800b54437ba866effb5', commit_message='Upload dataset', commit_description='', oid='8e9b9d74716c7135b62d2800b54437ba866effb5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/cheenshf/math_solution_FT_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='cheenshf/math_solution_FT_dataset'), pr_revision=None, pr_num=None)

In [17]:
test_formatted_texts = [
    format_training_example(ex["instruction"], ex["response"])
    for ex in test_data
]
test_dataset = Dataset.from_dict({"text": test_formatted_texts})
test_dataset.push_to_hub("cheenshf/math_solution_FT_dataset_test")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 10.4kB / 10.4kB            

CommitInfo(commit_url='https://huggingface.co/datasets/cheenshf/math_solution_FT_dataset_test/commit/000aa8428660a8c1b0b8cdc7f7c62e795a7c9bb7', commit_message='Upload dataset', commit_description='', oid='000aa8428660a8c1b0b8cdc7f7c62e795a7c9bb7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/cheenshf/math_solution_FT_dataset_test', endpoint='https://huggingface.co', repo_type='dataset', repo_id='cheenshf/math_solution_FT_dataset_test'), pr_revision=None, pr_num=None)

## Fine tuning deck

In [3]:
from datasets import load_dataset


train_dataset = load_dataset("cheenshf/math_solution_FT_dataset", split="train")
test_dataset = load_dataset("cheenshf/math_solution_FT_dataset_test", split="train")

print(f"Train examples: {len(train_dataset)}")
print(f"Test examples: {len(test_dataset)}")
print()
print(train_dataset[0]["text"][:300])

README.md:   0%|          | 0.00/267 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.3kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/266 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.4kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Train examples: 100
Test examples: 20

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

### Response:
Given


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],  # Wq, Wk, Wv, Wo
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [9]:
from trl import SFTTrainer, SFTConfig


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",       # matches the column name we created
    max_seq_length = max_seq_length,   # 2048, from earlier
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # small dataset (100 examples), a few passes over it
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        seed = 3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [10]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 2,162,688 of 496,195,456 (0.44% trained)


Step,Training Loss
1,0.558719
2,0.596346
3,0.487288
4,0.510405
5,0.488814
6,0.550364
7,0.479444
8,0.521234
9,0.577017
10,0.402884


In [13]:
def extract_instruction(formatted_text):
    start = formatted_text.index("### Instruction:\n") + len("### Instruction:\n")
    end = formatted_text.index("\n\n### Response:")
    return formatted_text[start:end].strip()

def extract_reference_response(formatted_text):
    start = formatted_text.index("### Response:\n") + len("### Response:\n")
    response = formatted_text[start:]
    return response.replace(tokenizer.eos_token, "").strip()

# Test on one example first to confirm parsing works
sample_text = test_dataset[0]["text"]
print("Extracted instruction:", extract_instruction(sample_text))
print()
print("Extracted reference response:", extract_reference_response(sample_text))

Extracted instruction: A craft store makes a third of its sales in the fabric section, a quarter of its sales in the jewelry section, and the rest in the stationery section. They made 36 sales today. How many sales were in the stationery section?

Extracted reference response: Given:
Total sales = 36
Fabric section = 1/3 of sales
Jewelry section = 1/4 of sales
Stationery section = remaining sales

Find:
Sales in the stationery section

Solution:

1. Fabric sales = 36 / 3 = 12
2. Jewelry sales = 36 / 4 = 9
3. Stationery sales = 36 - 12 - 9 = 15

Answer: 15


In [16]:
FastLanguageModel.for_inference(model)

test_instruction = extract_instruction(test_dataset[0]["text"])
prompt = alpaca_prompt.format(test_instruction, "")

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("INSTRUCTION:")
print(test_instruction)
print()
print("FINE-TUNED MODEL OUTPUT:")
print(response)
print()
print("REFERENCE (correct) ANSWER:")
print(extract_reference_response(test_dataset[0]["text"]))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUCTION:
A craft store makes a third of its sales in the fabric section, a quarter of its sales in the jewelry section, and the rest in the stationery section. They made 36 sales today. How many sales were in the stationery section?

FINE-TUNED MODEL OUTPUT:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
A craft store makes a third of its sales in the fabric section, a quarter of its sales in the jewelry section, and the rest in the stationery section. They made 36 sales today. How many sales were in the stationery section?

### Response:
Given:
Total sales = 36
Fabric section sales = 1/3 of total sales
Jewelry section sales = 1/4 of total sales

Find:
Stationery section sales

Solution:

1. Fabric section sales = 36 x 1/3 = 12
2. Jewelry section sales = 36 x 1/4 = 9
3. Stationery section sales = 36 - 12 - 9 = 15

Answer: 15

REFERENCE (correct) ANSWER:
Given:
Total sales = 36
Fabric section = 1/3 of sales

In [17]:
import re

def check_structure(response_text):
    """Check if response follows Given/Find/Solution/Answer, in order."""
    required = ["Given:", "Find:", "Solution:", "Answer:"]
    if not all(s in response_text for s in required):
        return False
    positions = [response_text.index(s) for s in required]
    return positions == sorted(positions)

def extract_final_answer(response_text):
    """Pull out the number after 'Answer:'"""
    match = re.search(r"Answer:\s*\$?(-?\d+[\d,]*\.?\d*)", response_text)
    if match:
        return match.group(1).replace(",", "")
    return None

results = []

FastLanguageModel.for_inference(model)

for i in range(len(test_dataset)):
    full_text = test_dataset[i]["text"]
    instruction = extract_instruction(full_text)
    reference_response = extract_reference_response(full_text)
    reference_answer = extract_final_answer(reference_response)

    prompt = alpaca_prompt.format(instruction, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # isolate just the generated response part (after "### Response:")
    if "### Response:" in full_output:
        model_response = full_output.split("### Response:")[-1].strip()
    else:
        model_response = full_output

    structure_pass = check_structure(model_response)
    model_answer = extract_final_answer(model_response)
    answer_pass = (model_answer == reference_answer) if model_answer else False

    results.append({
        "idx": i,
        "structure_pass": structure_pass,
        "answer_pass": answer_pass,
        "model_answer": model_answer,
        "reference_answer": reference_answer,
    })

    print(f"[{i}] structure={structure_pass}, answer={answer_pass} (got {model_answer}, expected {reference_answer})")

structure_rate = sum(r["structure_pass"] for r in results) / len(results)
answer_rate = sum(r["answer_pass"] for r in results) / len(results)

print()
print(f"Structure compliance: {structure_rate*100:.1f}% ({sum(r['structure_pass'] for r in results)}/{len(results)})")
print(f"Answer correctness: {answer_rate*100:.1f}% ({sum(r['answer_pass'] for r in results)}/{len(results)})")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[0] structure=True, answer=True (got 15, expected 15)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] structure=True, answer=False (got 4, expected 36)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[16] structure=True, answer=False (got 29, expected 21)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[17] structure=True, answer=False (got 8.66, expected 16)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[18] structure=True, answer=False (got 28, expected 13)
[19] structure=True, answer=False (got 1100, expected 140)

Structure compliance: 95.0% (19/20)
Answer correctness: 35.0% (7/20)


In [18]:
# Print full model output for a few failed examples, side by side with reference
failed_indices = [i for i, r in enumerate(results) if not r["answer_pass"]]

for idx in failed_indices[:5]:  # first 5 failures, adjust as needed
    full_text = test_dataset[idx]["text"]
    instruction = extract_instruction(full_text)
    reference_response = extract_reference_response(full_text)

    prompt = alpaca_prompt.format(instruction, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_response = full_output.split("### Response:")[-1].strip()

    print(f"===== FAILED IDX {idx} =====")
    print("INSTRUCTION:", instruction)
    print()
    print("MODEL OUTPUT:")
    print(model_response)
    print()
    print("REFERENCE:")
    print(reference_response)
    print("="*50)
    print()

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FAILED IDX 1 =====
INSTRUCTION: Marcy is a makeup artist and has agreed to do some makeup for her friend's wedding. The only makeup she has limited stock of is lip gloss so she counts how many tubes she needs. Each tube of lip gloss will hold enough lip gloss for 3 people's makeup. Marcy decides to bring 6 tubs of lip gloss, each of which holds 2 tubes of lip gloss, and this will be the exact amount she needs for everyone's makeup. How many people is Marcy painting with makeup?

MODEL OUTPUT:
Given:
Tubs of lip gloss = 6
Each tub has 2 tubes
Each tube holds enough for 3 people

Find:
Number of people Marcy is painting makeup for

Solution:

1. Total tubes of lip gloss = 6 x 2 = 12
2. Total people = 12 / 3 = 4

Answer: 4

REFERENCE:
Given:
Lip gloss per tube = enough for 3 people
Tubs of lip gloss = 6
Tubes per tub = 2

Find:
Number of people Marcy is doing makeup for

Solution:

1. Tubes of lip gloss = 6 x 2 = 12
2. People served = 12 x 3 = 36

Answer: 36



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FAILED IDX 3 =====
INSTRUCTION: Janice can type 6 sentences per minute. Today at work, Janice continued working on a paper she started typing yesterday. She typed for 20 minutes, took a break, and typed 15 minutes longer. She then had to erase 40 sentences she had typed incorrectly. After a meeting, she typed for 18 minutes more. In all, the paper had 536 sentences by the end of today. How many sentences did she start with today?

MODEL OUTPUT:
Given:
Janice's typing speed = 6 sentences per minute
Time spent typing = 20 + 15 + 18 = 53 minutes
Total sentences typed = 536

Find:
Number of sentences Janice started with today

Solution:

1. Total sentences typed = 536
2. Time spent typing = 53
3. Sentence per minute = 6
4. Time spent typing = 20 + 15 + 18 = 53
5. Sentence per minute = 536 / 53 = 10
6. Sentence typed = 10 x 20 = 200
7. Sentence typed = 200 + 15 = 215
8. Sentence typed = 215 x 6 = 1290
9. Sentence typed = 1290 - 40 = 1250
10. Sentence typed = 1250 x 6 = 7500
11. Senten

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FAILED IDX 6 =====
INSTRUCTION: John buys 10 packs of magic cards.  Each pack has 20 cards and 1/4 of those cards are uncommon.  How many uncommon cards did he get?

MODEL OUTPUT:
Given:
Packs = 10
Cards per pack = 20
Uncommon cards = 1/4 of total

Find:
Uncommon cards

Solution:

1. Uncommon cards = 20 x 1/4 = 5
2. Uncommon cards = 20 x 10 = 200

Answer: 200

REFERENCE:
Given:
Magic card packs = 10
Cards per pack = 20
Uncommon cards = 1/4 of each pack

Find:
Total uncommon cards

Solution:

1. Uncommon cards per pack = 20 / 4 = 5
2. Total uncommon cards = 10 x 5 = 50

Answer: 50



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FAILED IDX 7 =====
INSTRUCTION: There is very little car traffic on Happy Street. During the week, most cars pass it on Tuesday - 25. On Monday, 20% less than on Tuesday, and on Wednesday, 2 more cars than on Monday. On Thursday and Friday, it is about 10 cars each day. On the weekend, traffic drops to 5 cars per day. How many cars travel down Happy Street from Monday through Sunday?

MODEL OUTPUT:
Given:
Tuesday traffic = 25
Monday traffic = 20% less than Tuesday
Wednesday traffic = 2 more than Monday
Thursday and Friday traffic = 10 each
Weekend traffic = 5 each

Find:
Total cars per week

Solution:

1. Monday traffic = 25 x 20 / 100 = 5
2. Wednesday traffic = 5 + 2 = 7
3. Total traffic on Monday = 25 + 5 = 30
4. Total traffic on Wednesday = 7 + 30 = 37
5. Total traffic on Tuesday = 25 x 2 = 50
6. Total traffic on Monday and Wednesday = 50 + 37 = 87
7. Total traffic on Friday = 10 x 2 = 20
8. Total traffic on Monday, Wednesday, and Friday = 87 + 20 = 107
9. Total traffic on Sun